In [1]:
# MeXpose 2.0 Extension! Run within the MeXpose conda environment (mexpose_v2.yaml)

import io
import os
import re
import warnings

import imageio.v3 as iio
import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.colors as pcolors
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
from PIL import Image

from skimage.draw import polygon as sk_polygon
from scipy.spatial import ConvexHull

warnings.filterwarnings("ignore")
%matplotlib widget

# MeXpose Single-Cell Intra- vs Extracellular ROI Analysis
*Version Update 09.03.2026*

## Overview

This notebook extends the MeXpose single-cell analysis workflow by enabling spatial comparison of intracellular and extracellular metal distributions in LA-ICP-TOFMS imaging data.

Using reconstructed elemental images and corresponding cell segmentation masks, the workflow allows users to define spatial regions of interest (ROIs) interactively and compare intracellular and extracellular signal intensities.

## Method

1. A reconstructed elemental TIFF image and its corresponding cell segmentation mask are loaded.
2. ROIs are defined interactively via lasso/box selection.
3. The ROI polygon is converted into a pixel-level mask.
4. Pixels are classified as:
   - **Intracellular:** segmentation label > 0  
   - **Extracellular:** segmentation label = 0
5. Within each ROI the following quantities are computed:
   - number of intracellular pixels
   - number of extracellular pixels
   - summed signal intensity (counts) for intracellular pixels
   - summed signal intensity (counts) for extracellular pixels
6. Intra- vs extracellular signal distributions can be statistically compared.

## Inputs

- Reconstructed elemental TIFF image (e.g. Pt signal)
- Segmentation mask image with integer cell labels
- Single-cell feature table (optional for visualization)

## Outputs

For each ROI the workflow reports:

- intracellular pixel count
- extracellular pixel count
- summed intracellular signal
- summed extracellular signal
- intra/extracellular signal ratios

ROI statistics can optionally be exported as CSV files.

## Purpose

An interactive single-cell analysis tool to visualize and quantify intracellular vs. extracellular signal.

**Statistics:**

The stats export includes:

- number of cells (if single cell information is added)
- number of pixels in intracellular / extracellular regions
- sum of counts (intracellular and extracellular)
- intracellular/extracellular sum ratio

In [2]:
MAX_DISPLAY_WIDTH = 1850

# global

filtered_df = pd.DataFrame()  # loaded from CSV upload
PATH = ""  # set from working directory widget

# input widgets (CSV + directory)

csv_uploader = widgets.FileUpload(
    accept=".csv", multiple=False, description="Single-cell CSV (optional)",
    layout=widgets.Layout(width="auto"),
)
working_directory = widgets.Text(
    value="", placeholder="Enter your working directory path",
    description="Output Path:", style={"description_width": "initial"},
    layout=widgets.Layout(width="400px"),
)
csv_status = widgets.Output()

def on_csv_upload(change):
    global filtered_df
    
    if csv_uploader.value:
        uploaded_file = csv_uploader.value[0]
        filtered_df = pd.read_csv(io.BytesIO(uploaded_file.content))
        # channel dropdown options - just the TIFF intensity for now! change if needed
        channel_dropdown.options = ["TIFF intensity"] + list(filtered_df.columns)
        with csv_status:
            csv_status.clear_output(wait=True)
            print(f"Loaded CSV with {len(filtered_df)} rows, {len(filtered_df.columns)} columns")

def on_path_change(change):
    global PATH
    PATH = working_directory.value

# observers

csv_uploader.observe(on_csv_upload, names="value")
working_directory.observe(on_path_change, names="value")

In [ ]:

# visualization widgets

background_uploader = widgets.FileUpload(
    accept="image/*", multiple=False, description="Background Image (TIFF)",
    layout=widgets.Layout(width="auto"),
)
mask_uploader = widgets.FileUpload(
    accept="image/*", multiple=False, description="Segmentation Mask (e.g. from Cellpose)",
    layout=widgets.Layout(width="auto"),
)
space_view = widgets.RadioButtons(
    options=["Both", "Intracellular only", "Extracellular only"],
    value="Both", description="View:", style={"description_width": "initial"},
)
channel_dropdown = widgets.Dropdown(
    description="Channel",
    options=["TIFF intensity"],
    value="TIFF intensity",
)

op_slider = widgets.FloatSlider(
    value=0.6, min=0.05, max=1.0, step=0.05,
    description="Opacity", style={"description_width": "initial"},
    continuous_update=False,
)
range_slider = widgets.FloatRangeSlider(
    description="Intensity range", style={"description_width": "initial"},
    disabled=True, continuous_update=False,
)

name_text = widgets.Text(placeholder="e.g. Region_01", description="Save as:")
save_btn = widgets.Button(description="Save Selection", button_style="info", disabled=True)
save_help = widgets.HTML(
    value="<i>Save Selection: Exports a CSV to the output path containing all columns from your single-cell data for the selected cells. </i>"
)
selection_out = widgets.Output()
status_out = widgets.Output()
output_area = widgets.Output()

roi_name = widgets.Text(value="", placeholder="e.g. ROI_01", description="ROI Name:")
roi_stats_btn = widgets.Button(description="ROI intra/extra stats", button_style="success")
save_roi_fig_btn = widgets.Button(description="Save ROI figure", button_style="info")
roi_stats_out = widgets.Output()

# colorscale dropdowns (Plotly named colorscales; viridis and grey set as defaults)

plotly_cmaps = sorted(px.colors.named_colorscales())
intra_cmap_dd = widgets.Dropdown(
    description="Intra cmap", options=plotly_cmaps, value="viridis",
    style={"description_width": "initial"},
)
extra_cmap_dd = widgets.Dropdown(
    description="Extra cmap", options=plotly_cmaps, value="greys",
    style={"description_width": "initial"},
)

roi_help = widgets.HTML(
    value="<i>Use lasso/box select to define ROI. </i>"
)

ui = widgets.VBox(
    [
        widgets.HTML("<h3>1. Output Directory</h3>"),
        widgets.HBox([working_directory]),
        widgets.HTML("<h3>2. Load Data</h3>"),
        widgets.HBox([csv_uploader]),
        csv_status,
        widgets.HBox([background_uploader, mask_uploader]),
        widgets.HTML("<h3>3. Visualization Settings</h3>"),
        widgets.HBox([channel_dropdown, space_view, op_slider]),
        widgets.HBox([intra_cmap_dd, extra_cmap_dd]),
        widgets.HBox([range_slider]),
        roi_help,
        widgets.HBox([roi_name, roi_stats_btn, save_roi_fig_btn]),
        save_help,
        widgets.HBox([name_text, save_btn]),
        selection_out,
        roi_stats_out,
        status_out,
        output_area,
    ]
)

ui_out = widgets.Output()
with ui_out:
    ui_out.clear_output(wait=True)
    display(ui)
display(ui_out)

# state variables

last_slctd = []              # currently selected (unsaved) cell labels
saved_slct = {}              # saved selections
saved_traces = {}            # saved traces for plotting
last_roi_polygon = None      # from lasso selection callback
palette = px.colors.qualitative.Plotly

img_cache, int_cache, centroid_cache = {}, {}, {}
fig = None
last_channel = None
current_channel = None

# helpers

def prep_background(bg: np.ndarray) -> Image.Image:
    arr = bg.copy()
    if arr.dtype.kind == "f":
        peak = np.nanmax(arr) or 1.0
        arr = np.clip(arr / peak * 255, 0, 255).astype(np.uint8)
    elif arr.dtype.kind in "ui" and (arr.dtype.itemsize > 1 or arr.max() > 255):
        peak = arr.max() or 1
        arr = (arr / peak * 255).astype(np.uint8)

    if arr.ndim == 2:
        arr = np.stack([arr] * 3, axis=-1)
    elif arr.shape[-1] == 4:
        arr = arr[..., :3]
    return Image.fromarray(arr)

def get_cached(upl, key):
    if key not in img_cache:
        img_cache[key] = iio.imread(upl.value[0]["content"])
    return img_cache[key]

def build_centroids(mask):
    if "xy" in centroid_cache:
        return centroid_cache["xy"]
    ys, xs = np.nonzero(mask)
    df = pd.DataFrame({"x": xs, "y": ys, "lbl": mask[ys, xs]})
    cent = df.groupby("lbl")[["x", "y"]].mean().astype(int)
    cent = cent[cent.index != 0]
    centroid_cache["xy"] = cent
    return cent

def polyg_to_mask(polygon, shape_hw):
    if polygon is None:
        return None
    xs, ys = polygon
    H, W = shape_hw
    xs = np.clip(xs, 0, W - 1)
    ys = np.clip(ys, 0, H - 1)
    rr, cc = sk_polygon(ys, xs, shape=(H, W))
    m = np.zeros((H, W), dtype=bool)
    m[rr, cc] = True
    return m

def plotly_to_rgb255(c):
    c = str(c).strip()
    if c.startswith("#") and len(c) == 7:
        return (int(c[1:3], 16), int(c[3:5], 16), int(c[5:7], 16))
    m = re.match(r"rgb\(\s*([\d\.]+)\s*,\s*([\d\.]+)\s*,\s*([\d\.]+)\s*\)", c)
    if m:
        return (int(float(m.group(1))), int(float(m.group(2))), int(float(m.group(3))))
    return (255, 255, 255)

def apply_plotly(norm01, cs_name, alpha=255):
    n = 256
    cols = pcolors.sample_colorscale(cs_name, np.linspace(0, 1, n))
    lut = np.array([plotly_to_rgb255(c) for c in cols], dtype=np.uint8)
    H, W = norm01.shape
    rgba = np.zeros((H, W, 4), dtype=np.uint8)

    nanmask = np.isnan(norm01)
    idx = np.clip((np.nan_to_num(norm01) * (n - 1)).astype(np.int32), 0, n - 1)

    rgba[..., :3] = lut[idx]
    rgba[..., 3] = alpha
    rgba[nanmask] = [0, 0, 0, 0]
    return rgba

def roibbox_from_polygon(polygon, shape_hw):
    H, W = shape_hw
    if polygon is not None:
        xs, ys = polygon
        return (
            max(0, int(np.floor(xs.min()))),
            min(W, int(np.ceil(xs.max()))),
            max(0, int(np.floor(ys.min()))),
            min(H, int(np.ceil(ys.max()))),
        )
    return None

# selection callbacks

def on_sel(trace, pts, selector):
    global last_roi_polygon

    # capture lasso polygon if available
    try:
        if selector is not None and hasattr(selector, "xs") and hasattr(selector, "ys"):
            xs, ys = selector.xs, selector.ys
            if xs is not None and ys is not None and len(xs) >= 3:
                last_roi_polygon = (np.array(xs), np.array(ys))

        # box select fallback: use xrange/yrange rectangle as polygon
        elif selector is not None and hasattr(selector, "xrange") and hasattr(selector, "yrange"):
            xr, yr = selector.xrange, selector.yrange
            if xr is not None and yr is not None:
                xs = np.array([xr[0], xr[1], xr[1], xr[0]])
                ys = np.array([yr[0], yr[0], yr[1], yr[1]])
                last_roi_polygon = (xs, ys)
    except Exception:
        pass

    with selection_out:
        selection_out.clear_output(wait=True)

        idx = pts.point_inds
        trace.selectedpoints = idx or None

        if not idx:
            last_slctd.clear()
            save_btn.disabled = True
            print("No cells selected.")
            return

        labels = trace.customdata[idx, 0].astype(int).tolist()
        last_slctd[:] = labels
        save_btn.disabled = False

        vals = trace.customdata[idx, 1]
        print(f"{len(labels)} cells selected on '{current_channel}'.")
        print(f"min / max {current_channel}: {vals.min():.3g} / {vals.max():.3g}")

        if last_roi_polygon is not None:
            print(f"ROI polygon: {len(last_roi_polygon[0])} vertices captured")

# save selection
def save_selection(_):
    global filtered_df
    
    with selection_out:
        selection_out.clear_output(wait=True)

        if not PATH:
            print("Warning: set an output path first.")
            return

        name = name_text.value.strip()
        if not name:
            print("Warning: please enter a name for the selection first.")
            return
        if not last_slctd:
            print("Warning: no selection to save.")
            return
        if name in saved_slct:
            print(f"Warning: a selection called '{name}' already exists.")
            return
        if filtered_df.empty:
            print("Warning: no CSV data loaded.")
            return

        # ROI tagging, kept from singlecell_analysis.ipynb
        max_levels = 3              
        roi_cols = ["ROI", "secondary ROI", "tertiary ROI"]

        if roi_cols[0] not in filtered_df.columns:
            filtered_df[roi_cols[0]] = None

        sel_mask = filtered_df["cell_label"].isin(last_slctd)
        remaining = sel_mask.copy()

        for level, col in enumerate(roi_cols, start=1):
            if level > 1 and col not in filtered_df.columns:
                filtered_df[col] = None
            can_assign = remaining & filtered_df[col].isna()
            filtered_df.loc[can_assign, col] = name
            remaining = remaining & ~can_assign
            if not remaining.any():
                break

        if remaining.any():
            print(f"Warning: {int(remaining.sum())} row(s) already tagged at all {max_levels} ROI levels.")

        sub_df = filtered_df.loc[sel_mask].copy()
        saved_slct[name] = sub_df

        # scatter trace for saved selection
        colour = palette[len(saved_traces) % len(palette)]
        centroids = build_centroids(get_cached(mask_uploader, "mask"))
        coords = centroids.loc[last_slctd]

        sel_trace = go.Scattergl(
            x=coords.x, y=coords.y, mode="markers",
            marker=dict(size=6, symbol="circle", color=colour, line=dict(width=2)),
            name=name, showlegend=True, meta={"saved": True},
        )
        fig.add_trace(sel_trace)
        saved_traces[name] = sel_trace

        # export
        filepath = os.path.join(PATH, f"{name}.csv")
        sub_df.to_csv(filepath, index=False)
        print(f"Saved '{name}' with {len(sub_df)} rows to {filepath}")
        
        display(sub_df.head())

        save_btn.disabled = True
        name_text.value = ""

# roi stats + roi figure

def roi_stats(_):
    with roi_stats_out:
        roi_stats_out.clear_output(wait=True)

        if not PATH:
            print("Warning: set an output path first.")
            return

        try:
            img = get_cached(background_uploader, "bg").astype(np.float32)
            msk = get_cached(mask_uploader, "mask").astype(np.int32)
        except Exception:
            print("Warning: upload TIFF + mask first.")
            return

        if img.ndim != 2 or msk.shape != img.shape:
            print("Warning: TIFF/mask shape mismatch.")
            return

        # lasso/box polygon, else convex hull fallback
        roi_mask = polyg_to_mask(last_roi_polygon, img.shape)

        if roi_mask is None and last_slctd:
            print("No ROI polygon captured. Using convex hull of selected cells...")
            cent = build_centroids(msk)
            sel_cent = cent.loc[cent.index.isin(last_slctd)]
            if len(sel_cent) >= 3:
                pts = sel_cent[["x", "y"]].values
                try:
                    hull = ConvexHull(pts)
                    hp = pts[hull.vertices]
                    roi_mask = polyg_to_mask((hp[:, 0], hp[:, 1]), img.shape) # hull is computed from cell centroids, not cell boundaries
                except Exception as e:
                    print(f"Could not create convex hull: {e}")

        if roi_mask is None:
            print("Warning: no ROI available. Use lasso or box select to define an ROI first.")
            return

        intra_roi = (msk > 0) & roi_mask         # intra -> any labelled cell
        extra_roi = (msk == 0) & roi_mask        # extra -> unlabelled space

        intra_npix = int(np.count_nonzero(intra_roi))
        extra_npix = int(np.count_nonzero(extra_roi))

        # sums include all pixels regardless of sign; negative counts are not clipped
        intra_sum = float(np.nansum(img[intra_roi], dtype=np.float64))
        extra_sum = float(np.nansum(img[extra_roi], dtype=np.float64))

        # n of cells in ROI, note: centroid-based count only!
        n_cells = len(last_slctd) if last_slctd else 0

        name = roi_name.value.strip() or "ROI"
        ratio = intra_sum / extra_sum if extra_sum else np.nan

        df = pd.DataFrame([{
            "ROI": name,
            "n_cells": n_cells,
            "intra_pixels": intra_npix,
            "extra_pixels": extra_npix,
            "intra_sum_counts": intra_sum,
            "extra_sum_counts": extra_sum,
            "intra/extra_sum_ratio": ratio,
        }])

        display(df)

        out_path = os.path.join(PATH, f"{name}_intra_extra_stats.csv")
        df.to_csv(out_path, index=False)
        print(f"Saved: {out_path}")

def save_roi_fig(_):
    with roi_stats_out:
        roi_stats_out.clear_output(wait=True)

        if not PATH:
            print("Warning: set an output path first.")
            return

        try:
            img = get_cached(background_uploader, "bg")
            msk = get_cached(mask_uploader, "mask")
        except Exception:
            print("Warning: upload TIFF + mask first.")
            return

        if img.ndim != 2:
            print("Warning: background must be a 2D TIFF.")
            return

        name = roi_name.value.strip() or "ROI"
            
        out_path = os.path.join(PATH, f"{name}_figure.png")

        bbox = roibbox_from_polygon(last_roi_polygon, img.shape)
        if bbox is None:
            print("Warning: no ROI available. Use lasso or box select to define an ROI first.")
            return

        x_min, x_max, y_min, y_max = bbox

        pad = 20
        x_min = max(0, x_min - pad)
        x_max = min(img.shape[1], x_max + pad)
        y_min = max(0, y_min - pad)
        y_max = min(img.shape[0], y_max + pad)

        img_crop = img[y_min:y_max, x_min:x_max].astype(np.float32)
        msk_crop = msk[y_min:y_max, x_min:x_max]

        zmin, zmax = range_slider.value
        den = (zmax - zmin) if (zmax - zmin) != 0 else 1e-9
        norm = np.clip((img_crop - zmin) / den, 0, 1)

        intra_mask_crop = msk_crop > 0
        extra_mask_crop = msk_crop == 0
        view = space_view.value

        intra_cs = intra_cmap_dd.value
        extra_cs = extra_cmap_dd.value

        if view == "Both":
            rgba_intra = apply_plotly(norm, intra_cs, alpha=255)
            rgba_extra = apply_plotly(norm, extra_cs, alpha=180)
            rgba = np.zeros(msk_crop.shape + (4,), dtype=np.uint8)
            rgba[intra_mask_crop] = rgba_intra[intra_mask_crop]
            rgba[extra_mask_crop] = rgba_extra[extra_mask_crop]

        elif view == "Intracellular only":
            rgba = apply_plotly(norm, intra_cs, alpha=255)
            rgba[~intra_mask_crop] = [0, 0, 0, 0]

        else:   # extracellular only
            rgba = apply_plotly(norm, extra_cs, alpha=255)
            rgba[~extra_mask_crop] = [0, 0, 0, 0]

        bg_norm = np.clip((img_crop - zmin) / den, 0, 1)

        Hc, Wc = img_crop.shape
        fig, ax = plt.subplots(figsize=(10, 10 * Hc / Wc))

        ax.imshow(bg_norm, cmap="gray", vmin=0, vmax=1)
        ax.imshow(rgba, alpha=op_slider.value)

        # draw ROI outline (if polygon available)
        xs, ys = None, None
        if last_roi_polygon is not None:
            xs, ys = last_roi_polygon

        if xs is not None and ys is not None:
            # close the polygon by appending the first point to the end
            # this was added since the ROI borders were incomplete
            xs_closed = np.append(xs, xs[0])
            ys_closed = np.append(ys, ys[0])
            ax.plot(xs_closed - x_min, ys_closed - y_min, color="red", linewidth=2)

        ax.set_title(f"{name} ({view})", color="white", fontsize=14)
        ax.axis("off")
        fig.patch.set_facecolor("black")
        ax.set_facecolor("black")

        plt.tight_layout()
        fig.savefig(out_path, dpi=300, facecolor="black", bbox_inches="tight")
        plt.close(fig)

        print(f"Saved ROI figure: {out_path}")

# render

def render(full_update=True):
    global fig, last_channel, current_channel

    with status_out:
        status_out.clear_output(wait=True)
        print("Rendering....")

    if not full_update and fig is not None:
        fig.layout.images[1].opacity = op_slider.value
        return

    try:
        bg = get_cached(background_uploader, "bg")
        mask = get_cached(mask_uploader, "mask")
    except Exception:
        with output_area:
            output_area.clear_output(wait=True)
            print("Warning: upload images first.")
        return

    if bg.ndim != 2:
        with output_area:
            output_area.clear_output(wait=True)
            print(f"Warning: background TIFF must be 2D, got shape {bg.shape}")
        return

    if mask.shape != bg.shape:
        with output_area:
            output_area.clear_output(wait=True)
            print(f"Warning: shape mismatch - TIFF {bg.shape} vs mask {mask.shape}")
        return

    intra_mask = mask > 0
    extra_mask = mask == 0

    # update channel options if CSV is loaded

    if not filtered_df.empty:
        channel_dropdown.options = ["TIFF intensity"] + list(filtered_df.columns)
    
    ch = channel_dropdown.value or ""
    current_channel = ch
    if ch == "":
        return

    H, W = mask.shape
    disp_w = min(MAX_DISPLAY_WIDTH, W)
    disp_h = int(disp_w * (H / W))

    # continuous branch shown

    if ch == "TIFF intensity":
        inten = bg.astype(np.float32)
    else:
        if filtered_df.empty:
            with output_area:
                output_area.clear_output(wait=True)
                print("Please upload a single-cell CSV first.")
            return
            
        if ch in int_cache:
            inten = int_cache[ch]
        else:
            lut = filtered_df.set_index("cell_label")[ch].to_dict()
            table = np.full(mask.max() + 1, np.nan, dtype=float)
            for cid, val in lut.items():
                if cid < table.size:
                    table[cid] = val
            inten = table[mask]
            int_cache[ch] = inten

    if np.all(np.isnan(inten)):
        with output_area:
            output_area.clear_output(wait=True)
            print("No matching labels.")
        return

    nz = inten[~np.isnan(inten)]
    imin, imax = float(nz.min()), float(nz.max())

    if range_slider.disabled:
        range_slider.disabled = False
    range_slider.min, range_slider.max = imin, imax
    if ch != last_channel:
        range_slider.value = (imin, imax)
    zmin, zmax = range_slider.value
    last_channel = ch

    # intensity range is display-only; does not clip values used in stats
    den = (zmax - zmin) if (zmax - zmin) != 0 else 1e-9
    norm = np.clip((inten - zmin) / den, 0, 1)

    view = space_view.value
    intra_cs = intra_cmap_dd.value
    extra_cs = extra_cmap_dd.value

    if view == "Both":
        rgba_intra = apply_plotly(norm, intra_cs, alpha=255)
        rgba_extra = apply_plotly(norm, extra_cs, alpha=180)
        rgba = np.zeros(mask.shape + (4,), dtype=np.uint8)
        rgba[intra_mask] = rgba_intra[intra_mask]
        rgba[extra_mask] = rgba_extra[extra_mask]
    elif view == "Intracellular only":
        rgba = apply_plotly(norm, intra_cs, alpha=255)
        rgba[~intra_mask] = [0, 0, 0, 0]
    else:       # extracellular only
        rgba = apply_plotly(norm, extra_cs, alpha=255)
        rgba[~extra_mask] = [0, 0, 0, 0]

    overlay = Image.fromarray(rgba, mode="RGBA")

    cent = build_centroids(mask)
    if ch == "TIFF intensity":
        cent["val"] = cent.apply(lambda r: inten[int(r.y), int(r.x)], axis=1)
        hovertemplate = "cell %{customdata[0]}<br>TIFF intensity: %{customdata[1]:.3g}<extra></extra>"
    else:
        val_map = filtered_df.set_index("cell_label")[ch].to_dict()
        cent["val"] = cent.index.to_series().map(val_map.get)
        hovertemplate = f"cell %{{customdata[0]}}<br>{ch}: %{{customdata[1]:.3g}}<extra></extra>"

    # colorbar traces

    cb_args_extra = None
    if view == "Both":
        cb_args = dict(
            x=[W, W], y=[H, H], mode="markers",
            marker=dict(
                color=[zmin, zmax], colorscale=intra_cs, cmin=zmin, cmax=zmax,
                showscale=True,
                colorbar=dict(len=0.35, y=0.75, tickfont=dict(color="white"),
                              title=dict(text="Intra", font=dict(color="white"))),
                size=0
            ),
            hoverinfo="none", showlegend=False
        )
        cb_args_extra = dict(
            x=[W, W], y=[H, H], mode="markers",
            marker=dict(
                color=[zmin, zmax], colorscale=extra_cs, cmin=zmin, cmax=zmax,
                showscale=True,
                colorbar=dict(len=0.35, y=0.25, tickfont=dict(color="white"),
                              title=dict(text="Extra", font=dict(color="white"))),
                size=0
            ),
            hoverinfo="none", showlegend=False
        )
    else:
        cs = intra_cs if view == "Intracellular only" else extra_cs
        cb_args = dict(
            x=[W, W], y=[H, H], mode="markers",
            marker=dict(
                color=[zmin, zmax], colorscale=cs, cmin=zmin, cmax=zmax,
                showscale=True,
                colorbar=dict(len=0.7, y=0.5, tickfont=dict(color="white")),
                size=0
            ),
            hoverinfo="none", showlegend=False
        )

    def purge_extra_traces():
        # keep centroid scatter (trace 0) + saved traces
        keep = []
        for tr in fig.data:
            meta = tr.meta if isinstance(tr.meta, dict) else {}
            if tr is fig.data[0] or meta.get("saved", False):
                keep.append(tr)
        fig.data = tuple(keep)

    if fig is None:
        with output_area:
            output_area.clear_output(wait=True)

            fig = go.FigureWidget(
                layout=dict(
                    width=disp_w,
                    height=disp_h,
                    margin=dict(l=0, r=0, t=0, b=0),
                    dragmode="lasso",   # lasso/box selection for selecting cells
                    uirevision="keep",
                )
            )

            fig.update_layout(
                paper_bgcolor="black",
                plot_bgcolor="black",
                font=dict(color="white"),
            )

            # background + overlay images
            for src, layer in ((prep_background(bg), "below"), (overlay, "above")):
                fig.add_layout_image(
                    dict(
                        source=src, x=0, y=0, sizex=W, sizey=H,
                        layer=layer, xref="x", yref="y",
                        opacity=(op_slider.value if layer == "above" else 1),
                    )
                )

            # centroid scatter for selection / hover
            sc = go.Scattergl(
                x=cent.x, y=cent.y,
                mode="markers",
                marker=dict(size=3, opacity=0),
                customdata=np.stack([cent.index, cent["val"]], 1),
                hovertemplate=hovertemplate,
                selected=dict(marker=dict(opacity=1, size=6, color="white")),
                unselected=dict(marker=dict(opacity=0)),
                showlegend=False,
            )
            fig.add_trace(sc)
            fig.data[0].on_selection(on_sel)

            # colorbar traces
            fig.add_trace(go.Scatter(**cb_args))
            if cb_args_extra:
                fig.add_trace(go.Scatter(**cb_args_extra))

            fig.update_xaxes(visible=False, range=[0, W], scaleanchor="y")
            fig.update_yaxes(visible=False, range=[H, 0])

            display(fig)

    else:
        fig.layout.width, fig.layout.height = disp_w, disp_h
        fig.layout.images[0].source = prep_background(bg)
        fig.layout.images[1].source = overlay
        fig.layout.images[1].opacity = op_slider.value

        sc = fig.data[0]
        sc.x, sc.y = cent.x, cent.y
        sc.customdata = np.stack([cent.index, cent["val"]], 1)
        sc.hovertemplate = hovertemplate

        purge_extra_traces()
        fig.add_trace(go.Scatter(**cb_args))
        if cb_args_extra:
            fig.add_trace(go.Scatter(**cb_args_extra))

# wiring

save_btn.on_click(save_selection)
roi_stats_btn.on_click(roi_stats)
save_roi_fig_btn.on_click(save_roi_fig)

def on_opacity(change):
    render(full_update=False)

def on_other(change):
    render(full_update=True)

op_slider.observe(on_opacity, names="value")
for w in (channel_dropdown, range_slider, space_view, intra_cmap_dd, extra_cmap_dd):
    w.observe(on_other, names="value")

background_uploader.observe(on_other, names="value")
mask_uploader.observe(on_other, names="value")


Output()

# *Methods Notes:*

## Data Input
- Input: A reconstructed elemental TIFF image and its corresponding cell segmentation mask are loaded. When using the MeXpose pipeline, the TIFF images typically originate from the reconstruction notebooks `image_reconstruction_Nu.ipynb` or `image_reconstruction_TOFWerk.ipynb`. 
- **No normalization** is applied to pixel intensities prior to analysis.
- Segmentation masks: integer-labelled images where `label = 0` characterizes the extracellular space and `label > 0` the individual cell bodies.

## ROI Analysis
- ROIs are defined interactively via lasso or rectangular selection on the Plotly figure.
- The selected polygon is rasterized to a pixel-level binary mask using `skimage.draw.polygon`.
- Pixels within the ROI are classified as **intracellular** (`mask > 0`) or **extracellular** (`mask == 0`).

## Cell Counting
- `n_cells` = number of cell centroids within the user-defined ROI
- **Border cells** (cells intersected by the ROI boundary) are **not explicitly excluded** from pixel-level summation but may be partially included. Cell *count* only includes cells whose centroid lies within the selection.

*For rigorous border exclusion, consider applying an ROI erosion step before analysis.*

## Statistics
- Reported quantities per ROI:
  - `intra_pixels` / `extra_pixels`: pixel counts (not area-calibrated)
  - `intra_sum_counts` / `extra_sum_counts`: sum of pixel values (counts)
  - `intra/extra_sum_ratio`: ratio of summed intracellular to extracellular counts
- **No statistical testing** is performed within this notebook. Export the CSV and use R/Python for downstream tests (e.g. Mann-Whitney U, etc.). 

## Remarks
- Results depend on segmentation quality (Cellpose-SAM is usually used in house). 
- No correction for ion yield, ablation efficiency, or cell volume.
- Pixel size is not calibrated in this workflow; spatial metrics are in pixels.
- ROI polygon is user-defined (lasso or box select): Results are sensitive to ROI placement.